# Week 6: Spark Architecture and Data Processing

## Step 1: Creating Spark Session and Loading the Dataset

In this step, I created a Spark Session and loaded the Employee dataset into a Spark DataFrame. I used `header=True` so Spark reads the first row as column names and `inferSchema=True` so it automatically detects the correct data types.

In [1]:
from pyspark.sql import SparkSession

In [2]:
spark = SparkSession.builder \
    .appName("Employee Data Processing") \
    .getOrCreate()

In [4]:
df = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv("Employee.csv")

In [5]:
df.show(5)

+---------+-----------+---------+-----------+---+------+-----------+-------------------------+----------+
|Education|JoiningYear|     City|PaymentTier|Age|Gender|EverBenched|ExperienceInCurrentDomain|LeaveOrNot|
+---------+-----------+---------+-----------+---+------+-----------+-------------------------+----------+
|Bachelors|       2017|Bangalore|          3| 34|  Male|         No|                        0|         0|
|Bachelors|       2013|     Pune|          1| 28|Female|         No|                        3|         1|
|Bachelors|       2014|New Delhi|          3| 38|Female|         No|                        2|         0|
|  Masters|       2016|Bangalore|          3| 27|  Male|         No|                        5|         1|
|  Masters|       2017|     Pune|          3| 24|  Male|        Yes|                        2|         1|
+---------+-----------+---------+-----------+---+------+-----------+-------------------------+----------+
only showing top 5 rows


## Step 2: Checking the Schema

Before performing any transformations, I checked the schema of the dataset. This helps in understanding the data types of each column and ensures that Spark has correctly inferred the schema.

In [6]:
df.printSchema()

root
 |-- Education: string (nullable = true)
 |-- JoiningYear: integer (nullable = true)
 |-- City: string (nullable = true)
 |-- PaymentTier: integer (nullable = true)
 |-- Age: integer (nullable = true)
 |-- Gender: string (nullable = true)
 |-- EverBenched: string (nullable = true)
 |-- ExperienceInCurrentDomain: integer (nullable = true)
 |-- LeaveOrNot: integer (nullable = true)



## Step 3: Exploring the Dataset

Before applying any transformations, I explored the dataset to understand its size, columns, and sample records. This helps in getting a quick overview of the data and planning further processing.

In [7]:
df.count()

4653

In [8]:
print(df.columns)

['Education', 'JoiningYear', 'City', 'PaymentTier', 'Age', 'Gender', 'EverBenched', 'ExperienceInCurrentDomain', 'LeaveOrNot']


In [9]:
df.show(5)

+---------+-----------+---------+-----------+---+------+-----------+-------------------------+----------+
|Education|JoiningYear|     City|PaymentTier|Age|Gender|EverBenched|ExperienceInCurrentDomain|LeaveOrNot|
+---------+-----------+---------+-----------+---+------+-----------+-------------------------+----------+
|Bachelors|       2017|Bangalore|          3| 34|  Male|         No|                        0|         0|
|Bachelors|       2013|     Pune|          1| 28|Female|         No|                        3|         1|
|Bachelors|       2014|New Delhi|          3| 38|Female|         No|                        2|         0|
|  Masters|       2016|Bangalore|          3| 27|  Male|         No|                        5|         1|
|  Masters|       2017|     Pune|          3| 24|  Male|        Yes|                        2|         1|
+---------+-----------+---------+-----------+---+------+-----------+-------------------------+----------+
only showing top 5 rows


## Step 4: Checking for Null Values

Before cleaning the dataset, I checked whether any columns contain missing (null) values. This step helps identify data quality issues before applying transformations.

In [10]:
from pyspark.sql.functions import col, sum, when

In [11]:
df.select([
    sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in df.columns
]).show()

+---------+-----------+----+-----------+---+------+-----------+-------------------------+----------+
|Education|JoiningYear|City|PaymentTier|Age|Gender|EverBenched|ExperienceInCurrentDomain|LeaveOrNot|
+---------+-----------+----+-----------+---+------+-----------+-------------------------+----------+
|        0|          0|   0|          0|  0|     0|          0|                        0|         0|
+---------+-----------+----+-----------+---+------+-----------+-------------------------+----------+



## Step 5: Checking for Duplicate Records

After verifying the missing values, I checked the dataset for duplicate records. Removing duplicate data helps improve data quality and prevents incorrect analysis.

In [12]:
total_rows = df.count()
unique_rows = df.dropDuplicates().count()

print("Total Rows:", total_rows)
print("Unique Rows:", unique_rows)
print("Duplicate Rows:", total_rows - unique_rows)

Total Rows: 4653
Unique Rows: 2764
Duplicate Rows: 1889


In [13]:
df = df.dropDuplicates()

In [14]:
print("Rows after removing duplicates:", df.count())

Rows after removing duplicates: 2764


## Step 6: Filtering the Dataset

After cleaning the data, I applied a few filters to extract useful information from the dataset. Filtering helps in selecting only the required records for further analysis.

In [15]:
bangalore_df = df.filter(df.City == "Bangalore")

bangalore_df.show(5)

+---------+-----------+---------+-----------+---+------+-----------+-------------------------+----------+
|Education|JoiningYear|     City|PaymentTier|Age|Gender|EverBenched|ExperienceInCurrentDomain|LeaveOrNot|
+---------+-----------+---------+-----------+---+------+-----------+-------------------------+----------+
|      PHD|       2018|Bangalore|          3| 26|  Male|         No|                        4|         1|
|Bachelors|       2013|Bangalore|          3| 28|Female|         No|                        4|         1|
|Bachelors|       2018|Bangalore|          3| 26|  Male|        Yes|                        4|         1|
|Bachelors|       2012|Bangalore|          3| 29|  Male|         No|                        2|         1|
|Bachelors|       2016|Bangalore|          3| 28|  Male|        Yes|                        2|         0|
+---------+-----------+---------+-----------+---+------+-----------+-------------------------+----------+
only showing top 5 rows


In [16]:
tier3_df = df.filter(df.PaymentTier == 3)

tier3_df.show(5)

+---------+-----------+---------+-----------+---+------+-----------+-------------------------+----------+
|Education|JoiningYear|     City|PaymentTier|Age|Gender|EverBenched|ExperienceInCurrentDomain|LeaveOrNot|
+---------+-----------+---------+-----------+---+------+-----------+-------------------------+----------+
|      PHD|       2012|New Delhi|          3| 27|  Male|         No|                        5|         0|
|      PHD|       2018|Bangalore|          3| 26|  Male|         No|                        4|         1|
|Bachelors|       2013|Bangalore|          3| 28|Female|         No|                        4|         1|
|Bachelors|       2018|Bangalore|          3| 26|  Male|        Yes|                        4|         1|
|  Masters|       2015|New Delhi|          3| 29|  Male|         No|                        2|         0|
+---------+-----------+---------+-----------+---+------+-----------+-------------------------+----------+
only showing top 5 rows


In [17]:
age_df = df.filter(df.Age > 30)

age_df.show(5)

+---------+-----------+---------+-----------+---+------+-----------+-------------------------+----------+
|Education|JoiningYear|     City|PaymentTier|Age|Gender|EverBenched|ExperienceInCurrentDomain|LeaveOrNot|
+---------+-----------+---------+-----------+---+------+-----------+-------------------------+----------+
|Bachelors|       2017|New Delhi|          2| 39|  Male|         No|                        2|         0|
|Bachelors|       2018|     Pune|          3| 40|  Male|         No|                        0|         1|
|Bachelors|       2017|Bangalore|          3| 39|Female|         No|                        3|         0|
|Bachelors|       2015|Bangalore|          3| 31|  Male|         No|                        4|         0|
|Bachelors|       2017|New Delhi|          2| 33|  Male|         No|                        3|         0|
+---------+-----------+---------+-----------+---+------+-----------+-------------------------+----------+
only showing top 5 rows


## Step 7: Selecting Required Columns

Instead of working with all the columns, I selected only the required columns for analysis. This reduces unnecessary data processing and makes the output easier to understand.

In [18]:
selected_df = df.select(
    "Education",
    "City",
    "Age",
    "PaymentTier",
    "ExperienceInCurrentDomain"
)

selected_df.show(5)

+---------+---------+---+-----------+-------------------------+
|Education|     City|Age|PaymentTier|ExperienceInCurrentDomain|
+---------+---------+---+-----------+-------------------------+
|      PHD|New Delhi| 27|          3|                        5|
|      PHD|Bangalore| 26|          3|                        4|
|  Masters|     Pune| 24|          2|                        2|
|Bachelors|Bangalore| 28|          3|                        4|
|Bachelors|Bangalore| 26|          3|                        4|
+---------+---------+---+-----------+-------------------------+
only showing top 5 rows


## Step 8: Renaming a Column

To make the dataset more readable, I renamed one of the column names. Using shorter and meaningful column names makes the DataFrame easier to work with during analysis.

In [19]:
df = df.withColumnRenamed(
    "ExperienceInCurrentDomain",
    "CurrentDomainExperience"
)

In [20]:
df.columns

['Education',
 'JoiningYear',
 'City',
 'PaymentTier',
 'Age',
 'Gender',
 'EverBenched',
 'CurrentDomainExperience',
 'LeaveOrNot']

In [21]:
df.printSchema()

root
 |-- Education: string (nullable = true)
 |-- JoiningYear: integer (nullable = true)
 |-- City: string (nullable = true)
 |-- PaymentTier: integer (nullable = true)
 |-- Age: integer (nullable = true)
 |-- Gender: string (nullable = true)
 |-- EverBenched: string (nullable = true)
 |-- CurrentDomainExperience: integer (nullable = true)
 |-- LeaveOrNot: integer (nullable = true)



## Step 9: Casting a Data Type

The `PaymentTier` column was originally stored as an integer. I converted it to the `Double` data type to demonstrate type casting, which can be useful when performing mathematical calculations or working with decimal values.

In [22]:
df = df.withColumn(
    "PaymentTier",
    col("PaymentTier").cast("double")
)

In [23]:
df.printSchema()

root
 |-- Education: string (nullable = true)
 |-- JoiningYear: integer (nullable = true)
 |-- City: string (nullable = true)
 |-- PaymentTier: double (nullable = true)
 |-- Age: integer (nullable = true)
 |-- Gender: string (nullable = true)
 |-- EverBenched: string (nullable = true)
 |-- CurrentDomainExperience: integer (nullable = true)
 |-- LeaveOrNot: integer (nullable = true)



## Step 10: Adding a New Column

To make the dataset more informative, I created a new column named **ExperienceCategory** based on the employee's current domain experience. This helps classify employees into different experience levels for easier analysis.

In [24]:
df = df.withColumn(
    "ExperienceCategory",
    when(col("CurrentDomainExperience") <= 2, "Beginner")
    .when(col("CurrentDomainExperience") <= 5, "Intermediate")
    .otherwise("Experienced")
)

In [25]:
df.select(
    "CurrentDomainExperience",
    "ExperienceCategory"
).show(10)

+-----------------------+------------------+
|CurrentDomainExperience|ExperienceCategory|
+-----------------------+------------------+
|                      5|      Intermediate|
|                      4|      Intermediate|
|                      2|          Beginner|
|                      4|      Intermediate|
|                      4|      Intermediate|
|                      2|          Beginner|
|                      2|          Beginner|
|                      2|          Beginner|
|                      4|      Intermediate|
|                      5|      Intermediate|
+-----------------------+------------------+
only showing top 10 rows


## Step 11: Applying Transformations and Actions

Spark uses transformations to create a new DataFrame without executing the operation immediately. The execution starts only when an action is performed. In this step, I applied a transformation and then used actions to view the results.

In [26]:
experienced_df = df.filter(col("CurrentDomainExperience") > 3)

In [27]:
experienced_df.show(5)

+---------+-----------+---------+-----------+---+------+-----------+-----------------------+----------+------------------+
|Education|JoiningYear|     City|PaymentTier|Age|Gender|EverBenched|CurrentDomainExperience|LeaveOrNot|ExperienceCategory|
+---------+-----------+---------+-----------+---+------+-----------+-----------------------+----------+------------------+
|      PHD|       2012|New Delhi|        3.0| 27|  Male|         No|                      5|         0|      Intermediate|
|      PHD|       2018|Bangalore|        3.0| 26|  Male|         No|                      4|         1|      Intermediate|
|Bachelors|       2013|Bangalore|        3.0| 28|Female|         No|                      4|         1|      Intermediate|
|Bachelors|       2018|Bangalore|        3.0| 26|  Male|        Yes|                      4|         1|      Intermediate|
|      PHD|       2018|New Delhi|        2.0| 30|Female|         No|                      4|         1|      Intermediate|
+---------+-----

In [28]:
experienced_df.count()

912

In [29]:
print("Employees with more than 3 years of domain experience:", experienced_df.count())

Employees with more than 3 years of domain experience: 912


## Step 12: Saving the Processed Data

After completing the required transformations, I saved the processed dataset in both CSV and Parquet formats. CSV is easy to read and share, while Parquet is optimized for faster processing and better storage efficiency in big data applications.

In [30]:
df.write \
    .mode("overwrite") \
    .option("header", True) \
    .csv("output_csv")

In [31]:
df.write \
    .mode("overwrite") \
    .parquet("output_parquet")

# Step 13: Understanding Spark Architecture

Apache Spark follows a distributed architecture where different components work together to process large datasets efficiently.

### Driver

The Driver is the main program of a Spark application. It creates the Spark Session, converts the user code into execution tasks, and coordinates the overall execution.

### Cluster Manager

The Cluster Manager is responsible for allocating resources such as CPU and memory to the Spark application. It launches executors on the worker nodes.

### Executors

Executors run on worker nodes and execute the tasks assigned by the Driver. They also store intermediate data during processing and send the results back to the Driver.

The Driver, Cluster Manager, and Executors work together to execute Spark applications efficiently. This distributed architecture allows Spark to process large datasets much faster than traditional single-machine processing.

# Step 14: Lazy Evaluation and DAG

Spark does not execute transformations immediately. Instead, it records all the transformations and waits until an action is called. This approach is known as **Lazy Evaluation**.

When an action such as `show()` or `count()` is executed, Spark creates a **Directed Acyclic Graph (DAG)**. The DAG helps Spark optimize the execution plan before processing the data.

This optimization reduces unnecessary computations and improves the overall performance of the application.

During this task, operations such as `filter()`, `select()`, and `withColumn()` were executed only after actions like `show()` and `count()` were called. This demonstrates Spark's Lazy Evaluation and how it optimizes execution using a DAG.

# Step 15: File Formats and Performance

During this project, I worked with both CSV and Parquet file formats. Although both store the same data, their performance is different when processing large datasets in Spark.

### CSV vs Parquet

- **CSV** is a row-based file format. It is simple to read and easy to share, but it usually requires more storage space and takes longer to process large datasets.

- **Parquet** is a columnar file format. It stores data more efficiently, supports compression, and allows Spark to read only the required columns. This makes data processing faster and reduces storage requirements.

For this reason, I saved the final processed dataset in both CSV and Parquet formats.

### Predicate Pushdown

Parquet supports a feature called **Predicate Pushdown**. When a filter is applied, Spark reads only the required data instead of scanning the entire file. This reduces disk I/O, uses less memory, and improves query performance.

### Shuffle

A shuffle happens when Spark redistributes data between different partitions during operations such as `groupBy()` or `join()`. Since data needs to move across the cluster, shuffle is an expensive operation and can increase execution time. Therefore, it should be minimized whenever possible.

# Step 16: Spark Execution Modes

Spark applications can run in two different execution modes: **Client Mode** and **Cluster Mode**. The main difference between them is where the Driver program runs.

### Client Mode

In Client Mode, the Driver runs on the local machine from where the application is submitted. This mode is commonly used during development and testing because it allows the developer to monitor the application directly.

### Cluster Mode

In Cluster Mode, the Driver runs inside the cluster instead of the local machine. This mode is generally preferred for production environments because the application continues to run even if the client disconnects.

# Step 17: Conclusion

In this project, I learned how Apache Spark processes large datasets efficiently using DataFrames. I loaded the employee dataset, explored its schema, checked for missing values and duplicate records, applied filters, selected required columns, renamed a column, changed a data type, and created a new column for better analysis.

Finally, I saved the processed data in both CSV and Parquet formats. This project also helped me understand important Spark concepts such as Spark Architecture, Lazy Evaluation, DAG, Transformations, Actions, and different file formats.

## Performance Insights

- Spark automatically optimized the execution of transformations using Lazy Evaluation.
- Using `show()` and `count()` was a better choice than `collect()` because they avoid loading the entire dataset into the Driver's memory.
- Removing duplicate records improved the overall quality of the dataset.
- Saving the processed data in Parquet format provides better storage efficiency and faster query performance compared to CSV.
- Selecting only the required columns and applying filters helped reduce unnecessary data processing.